# LLM as a judge

In [1]:
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

import pandas as pd

In [2]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer"
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise"
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
load_dotenv()
openai_client = OpenAI()

In [6]:
df_answers = pd.read_csv("/workspaces/llm-zoomcamp-2026-code/Lesson 4B RAG and agent evaluation/data/rag-answers-new.csv")

In [7]:
answers = df_answers.to_dict(orient = "records")

In [8]:
rec = answers[0]
rec

{'question': 'Can I take this course at my own pace and still receive a certificate at the end?',
 'answer': 'No. You can only get a certificate if you finish the course with a live cohort; certificates are not awarded for self-paced mode.',
 'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'document': '69d122f12e'}

In [9]:
prompt = aqa_judge_prompt.format(
    question = rec["question"], 
    answer_orig = rec["answer_orig"], 
    answer_llm = rec["answer"])
print(prompt)

Question:
Can I take this course at my own pace and still receive a certificate at the end?

Original Answer (ground truth):
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

AI Answer:
No. You can only get a certificate if you finish the course with a live cohort; certificates are not awarded for self-paced mode.


In [10]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation
)
eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth on the core point: no certificate is available for self-paced mode, only for finishing with a live cohort. It omits the explanation about peer review, but that is additional detail rather than a required key point.', score='good')

In [11]:
calc_price(usage)

{'input_cost': 0.000261, 'output_cost': 0.000297, 'total_cost': 0.000558}

In [12]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [13]:
# Use evaluate function on a single record
eval_result, usage = evaluate_aqa(
    question = rec["question"], 
    answer_orig = rec["answer_orig"], 
    answer_llm = rec["answer"]
)

In [14]:
# Use evaluate function on a single record, but now as a function
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
# Costs 0.20 EUR
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

In [16]:
results[10]

({'question': 'I just found this course. Am I still allowed to join now, and does that affect my chance to get a certificate?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the core meaning: you may still join, but certificate eligibility depends on submitting the project while submissions are open. It adds the detail about finishing with the live cohort and joining too late, which is consistent with the original and not contradictory.'},
 ResponseUsage(input_tokens=329, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=67, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=396))

In [17]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [18]:
calc_total_price(usages)

0.20201849999999985

In [19]:
df_eval = pd.DataFrame(evaluations)

In [20]:
df_eval.head()

,question,document,score,reasoning
0,Can I take this course at my own pace and stil...,69d122f12e,good,The AI answer captures the key point: no certi...
1,Is a certificate available if I complete the c...,69d122f12e,good,The AI answer matches the ground truth: it cle...
2,Do self-paced learners get any certificate for...,69d122f12e,good,The AI answer matches the ground truth: it cle...
3,Why are certificates not issued for the self-p...,69d122f12e,good,The AI answer matches the ground truth: it sta...
4,Is peer review of capstone projects required i...,69d122f12e,bad,The AI answer is not semantically equivalent t...


In [21]:
df_eval.score.value_counts()

score
good    290
bad      25
Name: count, dtype: int64

In [22]:
df_eval.score.value_counts(normalize=True)

score
good    0.920635
bad     0.079365
Name: proportion, dtype: float64

In [23]:
df_eval.to_csv("data/rag-evaluations.csv", index=False)